In [ ]:
!pip install -q arabic-reshaper python-bidi

In [ ]:
import os, re, warnings, urllib.request
import cv2, numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms
from torchvision.models import inception_v3, Inception_V3_Weights
from transformers import BertModel, BertTokenizer
import arabic_reshaper
from bidi.algorithm import get_display

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.backends")
warnings.filterwarnings("ignore", message=".*Glyph.*missing from font.*")

# Setup Font & Paths
FONT_DIR, FONT_PATH = "fonts", "fonts/NotoNaskhArabic.ttf"
os.makedirs(FONT_DIR, exist_ok=True)
FONT_URL = "https://cdn.jsdelivr.net/gh/google/fonts@main/ofl/notonaskharabic/NotoNaskhArabic%5Bwght%5D.ttf"

if not os.path.exists(FONT_PATH) or os.path.getsize(FONT_PATH) < 1000:
    print("⏳ Downloading Noto Naskh Arabic font...")
    try:
        req = urllib.request.Request(FONT_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(FONT_PATH, "wb") as f:
            f.write(resp.read())
        print(f"✅ Font downloaded ({os.path.getsize(FONT_PATH)} bytes)")
    except Exception as e:
        print(f"❌ Font download failed: {e}")

urdu_font = fm.FontProperties(fname=FONT_PATH, size=12) if os.path.exists(FONT_PATH) and os.path.getsize(FONT_PATH) > 1000 else fm.FontProperties(family="sans-serif", size=12)
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXCEL_PATH = "/kaggle/input/datasets/ahmedali001/explainable-hate-paper/high_confidence_instances.xlsx"
IMG_BASE_DIR = "/kaggle/input/datasets/ahmedali001/explainable-hate-paper/high_confidence_instances"
CHECKPOINT_PATH = "/kaggle/input/models/ahmedali001/tensor-fusion-new/pytorch/default/1/tensor_model_multimodal_best_f1.pt"
OUTPUT_DIR = "xai_qualitative_individual"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Preprocessing & Model Setup
def clean_urdu(text):
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", str(text))
    return re.sub(r"[^\w\s]", " ", re.sub("[۰-۹]", " ", text)).replace("_", " ")

class TensorFusionMultimodalModel(nn.Module):
    def __init__(self, feature_dim=64, hidden_dropout=0.2):
        super().__init__()
        self.feature_dim = feature_dim
        self.bert = BertModel.from_pretrained("bert-base-multilingual-cased", attn_implementation="eager")
        self.bert_dropout = nn.Dropout(hidden_dropout)
        self.text_fc = nn.Sequential(nn.Linear(self.bert.config.hidden_size, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(hidden_dropout), nn.Linear(256, feature_dim))
        
        inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True)
        self.cnn_backbone = nn.Sequential(inc.Conv2d_1a_3x3, inc.Conv2d_2a_3x3, inc.Conv2d_2b_3x3, inc.maxpool1, inc.Conv2d_3b_1x1, inc.Conv2d_4a_3x3, inc.maxpool2, inc.Mixed_5b, inc.Mixed_5c, inc.Mixed_5d, inc.Mixed_6a, inc.Mixed_6b, inc.Mixed_6c, inc.Mixed_6d, inc.Mixed_6e, inc.Mixed_7a, inc.Mixed_7b, inc.Mixed_7c, inc.avgpool)
        self.image_fc = nn.Sequential(nn.Flatten(), nn.Linear(2048, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, feature_dim))
        
        self.fusion_fc = nn.Sequential(nn.Linear((feature_dim * feature_dim) + (2 * feature_dim) + 1, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 1))

    def tensor_fusion(self, text_feat, img_feat):
        text_feat, img_feat = F.normalize(text_feat, p=2, dim=1), F.normalize(img_feat, p=2, dim=1)
        outer = torch.bmm(text_feat.unsqueeze(2), img_feat.unsqueeze(1)).view(text_feat.size(0), -1)
        return torch.cat([outer, text_feat, img_feat, torch.ones(text_feat.size(0), 1, device=text_feat.device)], dim=1)

    def forward(self, input_ids, attention_mask, images):
        text_feat = self.text_fc(self.bert_dropout(self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output))
        img_feat = self.image_fc(self.cnn_backbone(images))
        return self.fusion_fc(self.tensor_fusion(text_feat, img_feat))

model = TensorFusionMultimodalModel()
state_dict = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
model.load_state_dict({k[7:] if k.startswith("module.") else k: v for k, v in state_dict.items()})
model.to(device).eval()

tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")
img_transform = transforms.Compose([transforms.Resize((299, 299)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def find_image_file(folder_path, tweet_id):
    if not os.path.exists(folder_path): return None
    str_id = str(tweet_id).strip()
    for f in os.listdir(folder_path):
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            name = os.path.splitext(f)[0]
            if str_id in name or name in str_id: return os.path.join(folder_path, f)
    return None

class TensorFusionGradCAM:
    def __init__(self, model):
        self.model, self.gradients, self.activations, self.hooks = model, None, None, []
        target = self.model.cnn_backbone[17]
        self.hooks.append(target.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach())))
        self.hooks.append(target.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach())))

    def generate_heatmap(self, input_ids, attention_mask, image_tensor):
        self.model.zero_grad()
        img_b = image_tensor.unsqueeze(0).to(device).requires_grad_()
        self.model(input_ids.unsqueeze(0).to(device), attention_mask.unsqueeze(0).to(device), img_b).backward()
        
        pooled_grads = torch.mean(self.gradients, dim=[0, 2, 3])
        acts = self.activations[0]
        for i in range(acts.shape[0]): acts[i, :, :] *= pooled_grads[i]
        
        heatmap = np.maximum(torch.mean(acts, dim=0).cpu().numpy(), 0)
        return heatmap / np.max(heatmap) if np.max(heatmap) != 0 else heatmap

    def remove_hooks(self):
        for h in self.hooks: h.remove()

def get_mbert_attention(full_model, input_ids, attention_mask):
    full_model.eval()
    orig_impl = getattr(full_model.bert.config, "_attn_implementation", "eager")
    full_model.bert.config._attn_implementation, full_model.bert.config.output_attentions = "eager", True
    
    with torch.no_grad():
        outputs = full_model.bert(input_ids=input_ids.unsqueeze(0).to(device), attention_mask=attention_mask.unsqueeze(0).to(device), output_attentions=True)
        valid_len = attention_mask.unsqueeze(0).sum().item()
        cls_attn = torch.mean(outputs.attentions[-1][0], dim=0)[0].cpu().numpy()[:valid_len]
        if np.max(cls_attn) > 0: cls_attn /= np.max(cls_attn)

    full_model.bert.config.output_attentions, full_model.bert.config._attn_implementation = False, orig_impl
    return cls_attn, valid_len

def overlay_gradcam(original_img_path, heatmap, alpha=0.5):
    raw = cv2.cvtColor(cv2.imread(original_img_path), cv2.COLOR_BGR2RGB) if original_img_path and os.path.exists(original_img_path) else np.full((299, 299, 3), 220, dtype=np.uint8)
    colored = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cv2.resize(heatmap, (raw.shape[1], raw.shape[0]))), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(raw, 1 - alpha, colored, alpha, 0)

def reconstruct_words(tokens, weights):
    words, word_weights, curr_w, curr_weights = [], [], "", []
    for t, w in zip(tokens, weights):
        if t in ["[CLS]", "[SEP]", "[PAD]"]: continue
        clean = re.sub(r"[\x00-\x1f\x7f-\x9f\ufffd\u200e\u200f]", "", t.replace("##", ""))
        if not clean: continue
        if t.startswith("##"):
            curr_w += clean
            curr_weights.append(w)
        else:
            if curr_w:
                words.append(curr_w)
                word_weights.append(np.mean(curr_weights))
            curr_w, curr_weights = clean, [w]
    if curr_w:
        words.append(curr_w)
        word_weights.append(np.mean(curr_weights))
    return words, word_weights

def draw_text_attention(ax, tokens, attn_weights):
    ax.axis("off")
    words, word_weights = reconstruct_words(tokens, attn_weights)
    reshaped = [get_display(arabic_reshaper.reshape(w.replace("\u0649", "\u0627").replace("\u0670", "\u0627"))) for w in words]
    
    ax.text(0.98, 0.95, ":mBERT [CLS] Attention Weights", fontsize=10, fontweight="bold", color="#333333", ha="right", va="top")
    x_pos, y_pos, line_h = 0.95, 0.87, 0.12
    for word, weight in zip(reshaped, word_weights):
        width = 0.03 + (len(word) * 0.028)
        if x_pos - width < 0.05: x_pos, y_pos = 0.95, y_pos - line_h
        bbox = dict(boxstyle="round,pad=0.3", facecolor=(1.0, 1.0 - weight * 0.75, 1.0 - weight * 0.75), edgecolor="none", alpha=0.85)
        ax.text(x_pos, y_pos, word, fontproperties=urdu_font, bbox=bbox, va="center", ha="right")
        x_pos -= width

def plot_text_score_barplot(tokens, attn_weights, title="Token Attention Scores", save_path=None, top_n=None):
    words, scores = reconstruct_words(tokens, attn_weights)
    reshaped = [get_display(arabic_reshaper.reshape(w.replace("\u0649", "\u0627").replace("\u0670", "\u0627"))) for w in words]
    
    reshaped, scores = np.array(reshaped[::-1]), np.array(scores[::-1])
    if top_n and top_n < len(reshaped):
        idx = np.argsort(np.abs(scores))[-top_n:]
        reshaped, scores = reshaped[idx], scores[idx]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    norm = plt.Normalize(vmin=min(scores) if len(scores) else 0, vmax=max(scores) if len(scores) else 1)
    bars = ax.bar(range(len(reshaped)), scores, color=plt.cm.Reds(norm(scores)), edgecolor="#475569", lw=0.8, width=0.6)
    
    ax.set_xticks(range(len(reshaped)))
    ax.set_xticklabels(reshaped, fontproperties=urdu_font, fontsize=12, rotation=45)
    ax.set_ylabel("Attention Weight", fontsize=11, fontweight="bold")
    ax.set_title(title, fontsize=12, fontweight="bold", pad=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2.0, h + (0.01 * (max(scores) if len(scores) else 1)), f"{h:.3f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Execution Pipeline
df = pd.read_excel(EXCEL_PATH)
df["Cleaned_Text"] = df["Text"].apply(clean_urdu)
grad_cam = TensorFusionGradCAM(model)

for cat in ["TP", "TN", "FP", "FN"]:
    sub_df = df[df["Category"] == cat]
    if sub_df.empty: continue

    row = sub_df.iloc[0]
    tweet_id, cleaned_text = row["Tweet_ID"], str(row["Cleaned_Text"])
    img_path = find_image_file(IMG_BASE_DIR, tweet_id)

    enc = tokenizer(cleaned_text, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
    input_ids, attention_mask = enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0)

    if img_path:
        image_tensor = img_transform(Image.open(img_path).convert("RGB"))
        raw_cv_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    else:
        image_tensor, raw_cv_img = torch.zeros((3, 299, 299)), np.full((299, 299, 3), 220, dtype=np.uint8)

    heatmap = grad_cam.generate_heatmap(input_ids, attention_mask, image_tensor)
    cam_overlay = overlay_gradcam(img_path, heatmap)

    # 1. Image Figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(raw_cv_img); ax1.axis("off"); ax1.set_title("Original Image", fontsize=11, fontweight="bold")
    ax2.imshow(cam_overlay); ax2.axis("off"); ax2.set_title("Grad-CAM Overlay", fontsize=11, fontweight="bold")
    plt.suptitle(f"[{cat}] Visual Explanation | Tweet ID: {tweet_id}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{cat}_image_explanation.png"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    # 2. Text Highlight Figure
    attn_weights, valid_len = get_mbert_attention(model, input_ids, attention_mask)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[:valid_len])

    fig, ax = plt.subplots(figsize=(8, 5))
    draw_text_attention(ax, tokens, attn_weights)
    ax.set_title(f"[{cat}] Qualitative Text Analysis\nTrue Label: {row['True_Label']} | Prob(Hate): {row['Model_Probability_Hate']:.4f}", fontsize=11, fontweight="bold", loc="left")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{cat}_text_explanation.png"), dpi=300, bbox_inches="tight")
    plt.show(); plt.close(fig)

    # 3. Bar Plot Figure
    plot_text_score_barplot(tokens=tokens, attn_weights=attn_weights, title=f"[{cat}] mBERT Token Attention Scores | Tweet ID: {tweet_id}", save_path=os.path.join(OUTPUT_DIR, f"{cat}_text_score_barplot.png"))

grad_cam.remove_hooks()
print(f"\n🎉 Process Complete! Gallery of images saved in '{OUTPUT_DIR}'.")